In [1]:
import pandas as pd
import numpy as np
import re
import datetime
from collections import Counter

In [2]:
# le fichier sur trouve a cette URL:
url = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe-v2-tertiaire-2/lines?size=10000&format=csv&after=10000%2C965634&header=true"

# on le charge directement dans une dataframe Pandas
data = pd.read_csv(url)
print(data.shape)

(10000, 63)


# Data exploration

In [3]:
data.head()

,N°DPE,Date_réception_DPE,Date_établissement_DPE,Date_visite_diagnostiqueur,Modèle_DPE,N°_DPE_remplacé,Date_fin_validité_DPE,Version_DPE,N°_DPE_immeuble_associé,Méthode_du_DPE,...,Type_énergie_n°2,Type_usage_énergie_n°2,Frais_annuel_énergie_n°2,Année_relève_conso_énergie_n°2,Conso_é_finale_énergie_n°3,Conso_é_primaire_énergie_n°3,Type_énergie_n°3,Type_usage_énergie_n°3,Frais_annuel_énergie_n°3,Année_relève_conso_énergie_n°3
0,2171T0890626E,2021-12-10,2021-12-09,2021-11-30,DPE 2006 tertiaire et ERP,NaN,2031-12-08,2.0,NaN,dpe tertiaire vierge,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2385T0962772Q,2023-03-23,2023-03-22,2023-03-20,DPE 2006 tertiaire et ERP,NaN,2033-03-21,2.2,NaN,dpe tertiaire facture,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2411T3209400P,2024-09-16,2024-09-15,2024-09-11,DPE 2006 tertiaire et ERP,2411T3208988T,2034-09-14,2.4,NaN,dpe tertiaire facture dans un bâtiment de loge...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2269T1391845U,2022-06-21,2022-06-20,2022-06-20,DPE 2006 tertiaire et ERP,NaN,2032-06-19,2.1,NaN,dpe tertiaire vierge,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2162T0808990Z,2021-11-30,2021-11-29,2021-10-25,DPE 2006 tertiaire et ERP,NaN,2031-11-28,1.1,NaN,dpe tertiaire vierge,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
data.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 63 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   N°DPE                              10000 non-null  object 
 1   Date_réception_DPE                 10000 non-null  object 
 2   Date_établissement_DPE             10000 non-null  object 
 3   Date_visite_diagnostiqueur         10000 non-null  object 
 4   Modèle_DPE                         10000 non-null  object 
 5   N°_DPE_remplacé                    743 non-null    object 
 6   Date_fin_validité_DPE              10000 non-null  object 
 7   Version_DPE                        10000 non-null  float64
 8   N°_DPE_immeuble_associé            28 non-null     object 
 9   Méthode_du_DPE                     9790 non-null   object 
 10  N°_immatriculation_copropriété     49 non-null     object 
 11  Invariant_fiscal_logement          0 non-null      floa

# Data preprocessing

## Rename columns

In [5]:
def rename_columns(columns):
    # en minuscule
    columns = [col.lower() for col in columns]
    # regex de remplacement
    rgxs = [
        (r"[°|/|']", "_"),
        (r"²", "2"),
        (r"[(|)]", ""),
        (r"é|è", "e"),
        (r"_+", "_"),
    ]
    # on remplace toutes les colonnes une par une
    for rgx in rgxs:
        columns = [re.sub(rgx[0], rgx[1], col) for col in columns]

    return columns

In [6]:
data.columns = rename_columns(data.columns)
data.columns

Index(['n_dpe', 'date_reception_dpe', 'date_etablissement_dpe',
       'date_visite_diagnostiqueur', 'modele_dpe', 'n_dpe_remplace',
       'date_fin_validite_dpe', 'version_dpe', 'n_dpe_immeuble_associe',
       'methode_du_dpe', 'n_immatriculation_copropriete',
       'invariant_fiscal_logement', 'etiquette_dpe', 'etiquette_ges',
       'conso_kwhep_m2_an', 'emission_ges_kgco2_m2_an', 'annee_construction',
       'categorie_erp', 'periode_construction', 'secteur_activite',
       'nombre_occupant', 'surface_shon', 'surface_utile',
       'type_energie_principale_chauffage', 'adresse_brute', 'nom_commune_ban',
       'code_insee_ban', 'n_voie_ban', 'identifiant_ban', 'adresse_ban',
       'code_postal_ban', 'score_ban', 'nom_rue_ban',
       'coordonnee_cartographique_x_ban', 'coordonnee_cartographique_y_ban',
       'code_postal_brut', 'n_etage_appartement', 'nom_residence',
       'complement_d_adresse_bâtiment', 'cage_d_escalier',
       'complement_d_adresse_logement', 'statut_geo

## Drop columns

In [7]:
columns_int = [
    "version_dpe",
    "surface_utile",
    "conso_kwhep_m2_an",
    "conso_e_finale_energie_n_1",
]

columns_categorical = [
    "periode_construction",
    "secteur_activite",
    "type_energie_principale_chauffage",
    "type_energie_n_1",
    "type_usage_energie_n_1",
]

target = 'etiquette_dpe'
id_col = 'n_dpe'

train_columns = columns_int + columns_categorical

In [8]:
train_columns

['version_dpe',
 'surface_utile',
 'conso_kwhep_m2_an',
 'conso_e_finale_energie_n_1',
 'periode_construction',
 'secteur_activite',
 'type_energie_principale_chauffage',
 'type_energie_n_1',
 'type_usage_energie_n_1']

In [9]:
data['version_dpe'] = data['version_dpe'] *10

for col in columns_int:
    data[col] = data[col].fillna(-1.0)
    data.loc[data[col] == "", col] = -1.0
    data[col] = data[col].astype(int)

In [10]:
data[columns_int].describe(percentiles = [0.99])

,version_dpe,surface_utile,conso_kwhep_m2_an,conso_e_finale_energie_n_1
count,10000.000000,10000.00000,10000.00000,1.000000e+04
mean,21.493800,453.23710,161.84710,6.495390e+04
std,3.405132,2950.58339,1230.48689,7.233381e+05
min,10.000000,1.00000,-933.00000,-1.000000e+00
50%,22.000000,100.00000,16.50000,6.970000e+02
99%,24.000000,6215.03000,1228.26000,1.064524e+06
max,24.000000,122867.00000,112336.00000,5.144700e+07


In [11]:
data = data[data['surface_utile'] < 9800]
data = data[data['conso_kwhep_m2_an'] < 2000]

In [12]:
for col in columns_categorical:
    data[col] = data[col].fillna("")

## Encoding categorical columns

In [13]:
data['type_energie_principale_chauffage'].value_counts()

type_energie_principale_chauffage
                                           8973
Électricité                                 643
Gaz naturel                                 208
Fioul domestique                             32
Réseau de Chauffage urbain                   32
autre combustible fossile                    11
Bois – Bûches                                 4
Bois – Granulés (pellets) ou briquettes       3
GPL                                           2
Propane                                       1
Name: count, dtype: int64

In [14]:
data['type_usage_energie_n_1'].value_counts(dropna=False)

type_usage_energie_n_1
                                      4650
périmètre de l'usage inconnu          4022
Chauffage                              806
Eau Chaude sanitaire                   211
Eclairage                               83
Ascenseur(s)                            62
Refroidissement                         29
Autres usages                           25
auxiliaires et ventilation               8
Bureautique                              7
Production d'électricité à demeure       6
Name: count, dtype: int64

In [15]:
map_type_energie = {
    "non renseigné": -1,
    "Électricité": 1,
    "Électricité d'origine renouvelable utilisée dans le bâtiment": 1,
    "Gaz naturel": 2,
    "Butane": 2,
    "Propane": 2,
    "GPL": 2,
    "Fioul domestique": 3,
    "Réseau de Chauffage urbain": 4,
    "Charbon": 5,
    "autre combustible fossile": 5,
    "Bois - Bûches": 6,
    "Bois - Plaquettes forestières": 6,
    "Bois - Granulés (pellets) ou briquettes": 6,
    "Bois - Plaquettes d'industrie": 6,
}

In [16]:
map_type_usage = {
    "non renseigné": -1,
    "périmètre de l'usage inconnu": -1,
    "Chauffage": 1,
    "Eau Chaude sanitaire": 2,
    "Eclairage": 3,
    "Refroidissement": 4,
    "auxiliaires et ventilation": 4,
    "Ascenseur(s)": 5,
    "Autres usages": 6,
    "Bureautique": 6,
    "Abonnements": 6,
    "Production d'électricité à demeure": 6,
}

In [17]:
map_secteur_activite = {
    "autres tertiaires non ERP": 1,
    "M : Magasins de vente, centres commerciaux": 2,
    "W : Administrations, banques, bureaux": 3,
    "locaux d'entreprise (bureaux)": 4,
    "J : Structures d’accueil pour personnes âgées ou personnes handicapées": 5,
    "N : Restaurants et débits de boisson": 6,
    "U : Établissements de soins": 7,
    "GHW : Bureaux": 8,
    "R : Établissements d’éveil, d’enseignement, de formation, centres de vacances, centres de loisirs sans hébergement": 9,
    "O : Hôtels et pensions de famille": 10,
    "GHZ : Usage mixte": 11,
    "X : Établissements sportifs couverts": 12,
    "L : Salles d'auditions, de conférences, de réunions, de spectacles ou à usage multiple": 13,
    "T : Salles d'exposition à vocation commerciale": 14,
    "P : Salles de danse et salles de jeux": 15,
    "GHR : Enseignement": 16,
    "V : Établissements de divers cultes": 17,
    "S : Bibliothèques, centres de documentation": 18,
    "OA : Hôtels-restaurants d'Altitude": 19,
    "GHU : Usage sanitaire": 20,
    "PA : Établissements de Plein Air": 21,
    "GHA : Habitation": 22,
    "GHO : Hôtel": 23,
    "Y : Musées": 24,
    "PS : Parcs de Stationnement couverts": 25,
    "GHTC : tour de contrôle": 26,
    "REF : REFuges de montagne": 27,
    "GA : Gares Accessibles au public (chemins de fer, téléphériques, remonte-pentes...)": 28,
    "CTS : Chapiteaux, Tentes et Structures toile": 29,
    "GHS : Dépôt d'archives": 30,
}

In [18]:
map_periode_construction = {
    "avant 1948": 0,
    "1948-1974": 1,
    "1975-1977": 2,
    "1978-1982": 3,
    "1983-1988": 4,
    "1989-2000": 5,
    "2001-2005": 6,
    "2006-2012": 7,
    "2013-2021": 8,
    "après 2021": 9,
}

In [19]:
map_target = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

In [20]:
def encode_categorical_with_map(data, column, mapping, default_unknown=""):
    # valeurs possibles
    valid_values = list(mapping.keys())
    # les valeurs inconnues
    data.loc[~data[column].isin(valid_values), column] = default_unknown
    # valeurs manquantes et inconnues
    mapping[default_unknown] = -1
    # encodage des valeurs connues
    data[column] = data[column].apply(lambda d: mapping[d])
    return data[column]

In [21]:
mappings = [
    map_periode_construction,
    map_secteur_activite,
    map_type_energie,
    map_type_usage,
    map_type_usage,
]

for col, mapping in zip(columns_categorical, mappings):
    data[col] = encode_categorical_with_map(data, col, mapping)

In [22]:
data[target] = encode_categorical_with_map(data, target, map_target)

In [23]:
all_columns = [id_col] + train_columns + [target]
data = data[all_columns]

data.reset_index(inplace = True, drop = True)

In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9909 entries, 0 to 9908
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   n_dpe                              9909 non-null   object
 1   version_dpe                        9909 non-null   int32 
 2   surface_utile                      9909 non-null   int32 
 3   conso_kwhep_m2_an                  9909 non-null   int32 
 4   conso_e_finale_energie_n_1         9909 non-null   int32 
 5   periode_construction               9909 non-null   int64 
 6   secteur_activite                   9909 non-null   int64 
 7   type_energie_principale_chauffage  9909 non-null   int64 
 8   type_energie_n_1                   9909 non-null   int64 
 9   type_usage_energie_n_1             9909 non-null   int64 
 10  etiquette_dpe                      9909 non-null   int64 
dtypes: int32(4), int64(6), object(1)
memory usage: 696.9+ KB


## Train test split

In [25]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

In [26]:
# # load data
# input_file = output_file
# data = pd.read_csv(input_file)
# # shuffle
# data = data.sample(frac=1, random_state=808).reset_index(drop=True)


In [27]:
data.columns

Index(['n_dpe', 'version_dpe', 'surface_utile', 'conso_kwhep_m2_an',
       'conso_e_finale_energie_n_1', 'periode_construction',
       'secteur_activite', 'type_energie_principale_chauffage',
       'type_energie_n_1', 'type_usage_energie_n_1', 'etiquette_dpe'],
      dtype='object')

In [28]:
# Assuming the last column is the target variable
X = data.iloc[:, :-1]  # Features
y = data.iloc[:, -1]  # Target variable
assert y.name == "etiquette_dpe"

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42
)

In [29]:
X_train = X_train.drop(columns=["n_dpe"])
id_test = X_test["n_dpe"]
X_test = X_test.drop(columns=["n_dpe"])

In [30]:
id_test

3050    2164T0627731O
6792    2438T1777809L
360     2276T0523837D
7223    2375T3929683F
5577    2371T1100717R
            ...      
4056    2375T1582251J
5314    2339T3043302G
373     2349T3911593L
9172    2206T2763569Z
9138    2437T0301527I
Name: n_dpe, Length: 3964, dtype: object

In [31]:
X_test

,version_dpe,surface_utile,conso_kwhep_m2_an,conso_e_finale_energie_n_1,periode_construction,secteur_activite,type_energie_principale_chauffage,type_energie_n_1,type_usage_energie_n_1
3050,11,100,0,0,1,6,-1,-1,-1
6792,23,100,41,8030,9,4,-1,-1,-1
360,20,95,370,15283,0,4,-1,-1,2
7223,23,100,-1,-1,1,7,-1,-1,-1
5577,22,100,-1,-1,0,2,-1,-1,-1
...,...,...,...,...,...,...,...,...,...
4056,22,100,686,5578,0,2,-1,-1,-1
5314,23,100,292,22516,0,6,-1,-1,-1
373,23,100,-1,-1,4,9,-1,-1,-1
9172,22,131,-1,-1,3,4,-1,-1,-1


In [32]:
rf = RandomForestClassifier()

# Define the parameter grid
param_grid = {
        "n_estimators": [200, 300],  # Number of trees
        "max_depth": [10],  # Maximum depth of the trees
        "min_samples_leaf": [1, 5],  # Maximum depth of the trees
}

# Setup GridSearchCV with k-fold cross-validation
cv = KFold(n_splits=3, random_state=84, shuffle=True)

grid_search = GridSearchCV(
    estimator=rf, param_grid=param_grid, cv=cv, scoring="accuracy", verbose=1
)

# Fit the model
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


GridSearchCV(cv=KFold(n_splits=3, random_state=84, shuffle=True),
             estimator=RandomForestClassifier(),
             param_grid={'max_depth': [10], 'min_samples_leaf': [1, 5],
                         'n_estimators': [200, 300]},
             scoring='accuracy', verbose=1)

In [33]:
    # Best parameters and best score
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_}")
    print(f"Best model: {grid_search.best_estimator_}")

    # Evaluate on the test set
    yhat = grid_search.predict(X_test)
    print(classification_report(y_test, yhat))



Best parameters: {'max_depth': 10, 'min_samples_leaf': 1, 'n_estimators': 300}
Best cross-validation score: 0.9069818676009375
Best model: RandomForestClassifier(max_depth=10, n_estimators=300)
              precision    recall  f1-score   support

          -1       1.00      1.00      1.00      1855
           1       0.95      0.85      0.90       215
           2       0.86      0.83      0.84       265
           3       0.91      0.84      0.87       581
           4       0.80      0.88      0.84       468
           5       0.75      0.89      0.81       265
           6       0.82      0.62      0.71       108
           7       0.87      0.90      0.88       207

    accuracy                           0.92      3964
   macro avg       0.87      0.85      0.86      3964
weighted avg       0.92      0.92      0.92      3964



In [34]:
# regroup into predictions dataframe
probabilities = grid_search.predict_proba(X_test)

In [35]:

predictions = pd.DataFrame()
predictions.index = id_test
predictions["prob"] = np.max(probabilities, axis=1)
predictions["yhat"] = yhat
predictions["y"] = y_test.values
print(predictions.head())


                   prob  yhat  y
n_dpe                           
2164T0627731O  0.997244     1  1
2438T1777809L  0.622094     2  2
2276T0523837D  0.661282     5  5
2375T3929683F  1.000000    -1 -1
2371T1100717R  1.000000    -1 -1


In [36]:
    # feature importance
    feature_importances = grid_search.best_estimator_.feature_importances_
    feature_names = X_train.columns

    # Create a dictionary mapping feature names to their importance
    importance_dict = dict(zip(feature_names, feature_importances))
    importance_dict = dict(
        sorted(importance_dict.items(), key=lambda item: item[1], reverse=True)
    )

    print(importance_dict)

{'conso_kwhep_m2_an': 0.607144026865584, 'conso_e_finale_energie_n_1': 0.2807467697593336, 'secteur_activite': 0.024879027643078745, 'surface_utile': 0.022406236045123346, 'version_dpe': 0.01987208748616172, 'type_usage_energie_n_1': 0.019120467274493767, 'periode_construction': 0.015200655822420917, 'type_energie_principale_chauffage': 0.0106307291038039, 'type_energie_n_1': 0.0}
